<a href="https://www.kaggle.com/code/soumyajitworkspace/transformer?scriptVersionId=307266428" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<centre>$$ Transformer Block$$</centre>

bolck is not possable bcuz you make full transformer then i will work so 

In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F 
import math
#_____________________________________________------attention 
def scaled_dot_product_attention(Q,K,V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q,K.transpose(-2,-1))
    scores = scores/math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0,-1e9)
    weights = F.softmax(scores,dim=-1)
    output = torch.matmul(weights,V)
    return output, weights
#class multi head attention 
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model,num_heads, dropout = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads 
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model,d_model)
        self.dropout = nn.Dropout(dropout)
    def split_heads(self, x,batch):
        x = x.view(batch,-1,self.num_heads,self.d_k)
        return x.transpose(1,2)
    def forward (self,Q,K,V, mask=None):
        batch = Q.size(0)
        
        Q = self.split_heads(self.W_Q(Q), batch)
        K = self.split_heads(self.W_K(K), batch)
        V = self.split_heads(self.W_V(V), batch)
        out, weigths = scaled_dot_product_attention(
            Q,K,V,mask
        )
        out = out.transpose(1,2).contiguous()
        out = out.view(batch,-1,self.d_model)
        out = self.W_O(out)
        return out,weigths
class feedforward(nn.Module):
    def __init__(self,d_model,d_ff,dropout= 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model,d_ff)
        self.linear2 = nn.Linear(d_ff,d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    def forward(self,x ):
        x = self.relu(self.linear1(x))
        x = self.dropout(x)
        x = self.linear2(x)
        return x
class Postionalencoding(nn.Module):
    def __init__(self,d_model,max_len=5000,dropout = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len,d_model)
        pos = torch.arange(0,max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)

        )
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    def forward(self,x ):
        x = x + self.pe[:, :x.size(1),:]
        return self.dropout(x)
    
class TransformerBlock(nn.Module):
    def __init__(self,d_model,num_heads,d_ff,dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ff = feedforward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x,mask=None):
        atten_out , _ = self.attention(x,x,x,mask)
        x = self.norm1(x + self.dropout(atten_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x
class transformerEncoder(nn.Module):
    def __init__(self, 
                 vocab_size,
                 d_model = 512,
                 num_heads = 8,
                 num_layers = 6,
                 d_ff = 2048,
                 max_len = 5000,
                 dropout = 0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,d_model)
        self.pos_enc = Postionalencoding(d_model,max_len,dropout)
        self.scale = math.sqrt(d_model)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model,num_heads,d_ff,dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    def forward(self,x,mask= None):
        x = self.embedding(x)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x,mask)
        return self.norm(x)
    
class TransformerClassifier(nn.Module):
    def __init__(self,vocab_size,num_classes,d_model=128,num_heads=8,num_layers=3,d_ff = 512):
        super().__init__()
        self.encoder = transformerEncoder(
            vocab_size = vocab_size,
            d_model = d_model,
            num_heads = num_heads,
            num_layers = num_layers,
            d_ff = d_ff
        )
        self.fc = nn.Linear(d_model,num_classes)
    def forward(self,x,mask=None):
        encoded = self.encoder(x,mask)
        cls_token = encoded.mean(dim=1)
        out = self.fc(cls_token)
        return out
device = "cuda" if torch.cuda.is_available() else "cpu"

vocab_size = 10000
batch_size = 8
seq_len    = 50

model = TransformerClassifier(
    vocab_size   = vocab_size,
    num_classes  = 2,       # pos/neg sentiment!
    d_model      = 128,
    num_heads    = 8,
    num_layers   = 3,
    d_ff         = 512
).to(device)

# Random input
x   = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
out = model(x)

print(f"Input shape:  {x.shape}")      # [8, 50]
print(f"Output shape: {out.shape}")    # [8, 2]
print(f"Parameters:   {sum(p.numel() for p in model.parameters()):,}")
print("Transformer works! ✅")




Input shape:  torch.Size([8, 50])
Output shape: torch.Size([8, 2])
Parameters:   1,875,330
Transformer works! ✅
